## Jason_3 Satellite


In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from xgboost.callback import EarlyStopping
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

In [2]:
import pandas as pd

# Replace with appropriate path 
df_tles = pd.read_csv('/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/Jason-3.csv', 
                      index_col=0, parse_dates=True)

# Step 2: Check if index is timezone-naive or timezone-aware
if df_tles.index.tz is None:
    df_tles.index = df_tles.index.tz_localize('UTC')
else:
    df_tles.index = df_tles.index.tz_convert('UTC')

# Step 3: Now you can safely continue
print(df_tles.describe())
print(df_tles.index.inferred_type)


       eccentricity  argument of perigee  inclination  mean anomaly  \
count   2410.000000          2410.000000  2410.000000   2410.000000   
mean       0.000789             4.713344     1.152634     -4.713325   
std        0.000039             0.048556     0.000029      0.049303   
min        0.000495             4.522810     1.152570     -5.138166   
25%        0.000767             4.676146     1.152610     -4.756978   
50%        0.000790             4.711369     1.152636     -4.711379   
75%        0.000813             4.757195     1.152654     -4.675909   
max        0.001822             4.925965     1.152734     -4.521766   

       Brouwer mean motion  right ascension  
count          2410.000000      2410.000000  
mean              0.055906         3.137078  
std               0.000010         1.813543  
min               0.055691         0.002424  
25%               0.055907         1.570272  
50%               0.055907         3.127959  
75%               0.055907         4.7

### Extract and scale Brouwer mean motion

In [3]:
df_element_1 = df_tles[["Brouwer mean motion"]]
df_element_1 = (df_element_1 - df_element_1.mean())*1e7
df_element_1.describe()


,Brouwer mean motion
count,2.410000e+03
mean,5.032864e-11
std,1.010428e+02
min,-2.149569e+03
25%,4.860613e+00
50%,5.065401e+00
75%,5.416743e+00
max,1.692533e+03


### Visualize the scaled element

In [4]:
import plotly.graph_objects as go


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_element_1.index, 
    y=df_element_1[df_element_1.columns[0]],  
    mode="lines",  
    name="Orbital Element"
))

fig.update_layout(
    title="Brouwer mean motion Over Time",
    xaxis_title="Time",
    yaxis_title="Brouwer mean motion",
    template="plotly",
    width=1400
)

fig.show()


### Create lag features for time series forecasting

In [5]:
NUM_LAG_FEATURES = 3

df_y = df_element_1.copy()
df_x = df_element_1.shift(1).rename(columns={"Brouwer mean motion": "bmm_lag_1"})

for lag in range(2, NUM_LAG_FEATURES + 1):
    df_x[f"bmm_lag_{lag}"] = df_element_1.shift(lag)

# Drop rows with NaNs
df_x = df_x.iloc[NUM_LAG_FEATURES:]
df_y = df_y.iloc[NUM_LAG_FEATURES:]


### Split for hyperparameter tuning


In [6]:
# Split for tuning
split_index = int(len(df_x) * 0.8)
split_date = df_x.index[split_index].strftime("%Y-%m-%d")

df_x_train = df_x[:split_date]
df_y_train = df_y[:split_date]
df_x_test = df_x[split_date:]
df_y_test = df_y[split_date:]

# Tune XGBoost model with early stopping
tuned_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42
)

tuned_model.fit(
    df_x_train,
    df_y_train.values.ravel(),
    eval_set=[(df_x_train, df_y_train), (df_x_test, df_y_test)],
    verbose=True
)

# View best iteration and RMSE
best_n_estimators = tuned_model.best_iteration + 1  
print(f"Best number of trees: {best_n_estimators}")


[0]	validation_0-rmse:40.68709	validation_1-rmse:183.27819
[1]	validation_0-rmse:39.15999	validation_1-rmse:181.23763
[2]	validation_0-rmse:37.77338	validation_1-rmse:179.39711
[3]	validation_0-rmse:36.51530	validation_1-rmse:178.48034
[4]	validation_0-rmse:35.37472	validation_1-rmse:177.74747
[5]	validation_0-rmse:34.34144	validation_1-rmse:177.17362
[6]	validation_0-rmse:33.40605	validation_1-rmse:176.73563
[7]	validation_0-rmse:32.55987	validation_1-rmse:176.41442
[8]	validation_0-rmse:31.79490	validation_1-rmse:176.19262
[9]	validation_0-rmse:31.10376	validation_1-rmse:176.05562
[10]	validation_0-rmse:30.47969	validation_1-rmse:175.99036
[11]	validation_0-rmse:29.91648	validation_1-rmse:175.98533
[12]	validation_0-rmse:29.40843	validation_1-rmse:176.03102
[13]	validation_0-rmse:28.95033	validation_1-rmse:176.11866
[14]	validation_0-rmse:28.53743	validation_1-rmse:176.24101
[15]	validation_0-rmse:28.16538	validation_1-rmse:176.39200
[16]	validation_0-rmse:27.83024	validation_1-rmse:

### Retrain final model on full dataset

In [7]:
# Use full data for final model
df_x_full = df_x.copy()
df_y_full = df_y.copy()

final_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42
)

final_model.fit(df_x_full, df_y_full.values.ravel())


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

### Make predictions on full data and compute residuals

In [8]:
# Predict and calculate residuals
y_pred_full = final_model.predict(df_x_full)
residuals_full = y_pred_full - df_y_full["Brouwer mean motion"].values

df_result = df_y_full.copy()
df_result["predicted"] = y_pred_full
df_result["residuals"] = residuals_full


###  Plot Observed vs Predicted (Full Time Range)

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines',
    name='Observed'
))

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines',
    name='Predicted'
))

fig.update_layout(
    title='Observed vs Predicted Brouwer Mean Motion (Full Series)',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    template='plotly_white',
    width=1200
)

fig.show()


### Plot residuals over time 

In [10]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals'
))

fig.add_hline(y=0, line_dash="dot", line_color="black")

fig.update_layout(
    title='Residuals Over Time',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width= 1400
)

fig.show()


### Plot Residuals with Ground Truth Maneuvers

In [11]:
# Load maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_JASON-3.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

# Localize to utc for non fengyun
for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")

# Filter to full range
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual range
y_min = df_result["residuals"].min()
y_max = df_result["residuals"].max()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

# Ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    hover_text = f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[y_min, y_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[hover_text, hover_text],
        showlegend=False
    ))

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.update_layout(
    title='Residuals with Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()


#### Detect Anomalies (Using 3σ Rule)

In [12]:
import numpy as np

# Calculate mean and standard deviation of residuals
residual_mean = df_result["residuals"].mean()
residual_std = df_result["residuals"].std()
 
# Define upper and lower thresholds
upper = residual_mean + 2 * residual_std
lower = residual_mean - 2 * residual_std

# Flag anomalies: residuals that fall outside the threshold range
df_result["anomaly"] = (df_result["residuals"] > upper) | (df_result["residuals"] < lower)

print(f"Anomaly threshold range: {lower:.4f} to {upper:.4f}")
print(" Anomaly timestamps:")
print(df_result[df_result["anomaly"]].index)


Anomaly threshold range: -80.0086 to 80.0210
 Anomaly timestamps:
DatetimeIndex(['2016-02-07 01:39:21.000384+00:00',
               '2016-02-10 12:10:21.869184+00:00',
               '2016-11-18 21:47:39.474527+00:00',
               '2017-11-10 13:24:54.260640+00:00',
               '2018-07-02 13:15:37.974240+00:00',
               '2018-10-31 21:10:49.550879+00:00',
               '2019-02-24 19:48:23.918976+00:00',
               '2020-12-08 05:27:51.590879+00:00',
               '2021-04-23 03:47:18.731615+00:00',
               '2021-04-29 11:34:05.209535+00:00',
               '2022-01-10 11:49:54.812927+00:00',
               '2022-01-11 14:03:55.438848+00:00',
               '2022-04-15 20:19:09.177599+00:00',
               '2022-04-18 12:15:09.506016+00:00',
               '2022-04-19 10:47:05.310527+00:00',
               '2022-04-20 13:02:43.802591+00:00',
               '2022-04-21 11:32:09.824063+00:00'],
              dtype='datetime64[ns, UTC]', freq=None)


### Plotting detected anomalies vs ground truth

In [13]:

ground_truth_df = pd.read_csv("/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_JASON-3.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)
for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")

# Filter maneuver data to match df_result range 
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual y-axis range for drawing vertical maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Plot observed values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines+markers',
    name='Observed',
    marker=dict(size=4),
), secondary_y=False)

# Plot predicted values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines+markers',
    name='Predicted',
    marker=dict(size=4),
), secondary_y=False)

# Plot residuals 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
), secondary_y=True)

# Plot detected anomalies 
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}',
), secondary_y=True)

# Plot ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left",
    secondary_y=True
)
fig.update_layout(
    title='Observed vs Predicted with Residuals, Detected Anomalies, and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    yaxis2_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[test_start, test_end]
)

fig.show()


### Residuals with detected anomaly and ground truth maneuver

In [14]:

# Residual range for drawing maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()

# Create figure 
fig = go.Figure()

# Residuals
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    line=dict(color='blue')
))

#  Detected Anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

# Ground Truth Maneuver Lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

fig.update_layout(
    title='Residuals with Detected Anomalies and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[df_result.index.min(), df_result.index.max()]
)

fig.show()
